In [95]:
from openai import OpenAI
import gradio as gr
import os
import json
import pprint

from dotenv import load_dotenv


In [97]:
# ollama_client = OpenAI(
#     base_url="http://127.0.0.1:11434/v1",
#     api_key="ollama"
# )


load_dotenv()  # Load environment variables from .env file

gpt_model = "gpt-4o-mini"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
mistral_model = "mistral:latest"
llama_model = "llama3.2:1b"
tiny_model = "tinyllama:latest"
neural_model = "neural-chat:latest"


In [98]:
system_prompt = """
You are a customer support assistant for an airline called Flight AI.
Your only capability is to check ticket prices for destinations.

Rules:
1. Respond in no more than one short, courteous sentence.
2. If the user greeting is generic (e.g., "Hi", "Hello", "Hey"), respond with: "Hi, I am your Flight Assistant, how may I help you?"
3. If the user asks about ticket prices, ask for the destination city if not provided.
4. Use the ticket_price_checker tool to look up prices for the destination.
5. If you truly do not know the answer, say "I don't know".
"""

curr_model = gpt_model

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}]
    
    for h in history:
        messages.append({
            "role": h["role"],
            "content": h["content"]
        })
    
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=llama_model,
        messages=messages,
        temperature=0
        )
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Adding chat_tooling details:


In [99]:
ticket_price = {
    'london': '$500',
    'paris': '$450',
    'new york': '$700',
    'tokyo': '$800',
    'sydney': '$900'
}

In [100]:
def ticket_price_checker(destination_city):
    
    print("Inside the ticket price checker function")
    print(f"Received destination city: {destination_city}")
    if not destination_city:
        return "No destination city provided."

    # Make lowercase safely
    destination_city = str(destination_city).lower()

    price = ticket_price.get(destination_city, None)
    if price:
        return f"The ticket price to {destination_city.title()} is {price}."
    else:
        return "I don't know the ticket price for that destination."

In [102]:
ticket_price_checker("Tokyo")

Inside the ticket price checker function
Received destination city: Tokyo


'The ticket price to Tokyo is $800.'

In [103]:
ticket_price_checker_function = {
    "name": "ticket_price_checker",
    "description": "Checks the ticket price for a given destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The name of the destination city to check the ticket price for."
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [104]:
tools = [{"type": "function", "function": ticket_price_checker_function}]
tools

[{'type': 'function',
  'function': {'name': 'ticket_price_checker',
   'description': 'Checks the ticket price for a given destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The name of the destination city to check the ticket price for.'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [105]:
def handle_tool_call(message):

    print("Handling tool call...")
    print(f"Message received for tool call: \n{message}")
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "ticket_price_checker":
        arguments = json.loads(tool_call.function.arguments)
        destination_city = arguments.get("destination_city", "")
        print(f"Tool call arguments: {arguments}")
        price = ticket_price_checker(destination_city)
        
        response = {
            "role": "tool",
            "content": price,
            "tool_call_id": tool_call.id
            }
        return response
    else:
        return {
            "role": "tool",
            "content": "Unknown tool called.",
            "tool_call_id": tool_call.id
        }
    

In [ ]:
def tooling_chat_v1(message, history):

    print("Inside the tooling_chat function")
    messages = [{"role": "system", "content": system_prompt}]
    
    for h in history:
        messages.append({
            "role": h["role"],
            "content": h["content"]
        })
    
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=curr_model,
        messages=messages,
        temperature=0,
        tools=tools,
        tool_choice="auto"
    )
    
    if response.choices[0].finish_reason == "tool_calls":
        messages.append({
        "role": "assistant",
        "content": response.choices[0].message.content,
        "tool_calls": response.choices[0].message.tool_calls
        })
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(response)

        for msg in messages:
            print(msg)

        response = client.chat.completions.create(
            model=curr_model,
            messages=messages
        )

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=tooling_chat_v1).launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


Inside the tooling_chat function
Inside the tooling_chat function
Inside the tooling_chat function
Handling tool call...
Message received for tool call: 
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_5pBidGfE6xXIxI8EVJW2OY5B', function=Function(arguments='{"destination_city":"Paris"}', name='ticket_price_checker'), type='function')])
Tool call arguments: {'destination_city': 'Paris'}
Inside the ticket price checker function
Received destination city: Paris


Handle multiple tool calls:

In [112]:
def handle_tool_calls(message):

    print("Handling tool calls...")
    print(f"Message received for tool call: \n{message}")
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "ticket_price_checker":
            arguments = json.loads(tool_call.function.arguments)
            destination_city = arguments.get("destination_city", "")
            print(f"Tool call arguments: {arguments}")
            price = ticket_price_checker(destination_city)
            
            response = {
                "role": "tool",
                "content": price,
                "tool_call_id": tool_call.id
                }
            responses.append(response)
        else:
            responses.append({
                "role": "tool",
                "content": "Unknown tool called.",
                "tool_call_id": tool_call.id
            })
    return responses

In [111]:
def tooling_chat_v2(message, history):

    print("Inside the tooling_chat function")
    messages = [{"role": "system", "content": system_prompt}]
    
    for h in history:
        content = h["content"]
        if isinstance(content, list):
            content = content[0]["text"] if content else ""
        messages.append({
            "role": h["role"],
            "content": content
        })
    
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=curr_model,
        messages=messages,
        temperature=0,
        tools=tools,
        tool_choice="auto"
    )
    
    if response.choices[0].finish_reason == "tool_calls":
        messages.append({
        "role": "assistant",
        "content": response.choices[0].message.content,
        "tool_calls": response.choices[0].message.tool_calls
        })
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.extend(responses)

        for msg in messages:
            print(msg)

        response = client.chat.completions.create(
            model=curr_model,
            messages=messages
        )

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=tooling_chat_v2).launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


Inside the tooling_chat function
Inside the tooling_chat function
Inside the tooling_chat function
Handling tool calls...
Message received for tool call: 
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BECi9XPaZ8OQoIshjjwrMPpF', function=Function(arguments='{"destination_city": "Paris"}', name='ticket_price_checker'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_h8P49gJHeMO6Q8cCY25lnT9y', function=Function(arguments='{"destination_city": "Tokyo"}', name='ticket_price_checker'), type='function')])
Tool call arguments: {'destination_city': 'Paris'}
Inside the ticket price checker function
Received destination city: Paris
Tool call arguments: {'destination_city': 'Tokyo'}
Inside the ticket price checker function
Received destination city: Tokyo
{'role': 'system', 'content': '\nYou are a customer support assistant for an airline called Flight A

Multiple Tool_Calls with Sequential call support:

In [116]:
def handle_tool_calls(message):

    print("Handling tool calls...")
    print(f"Message received for tool call: \n{message}")
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "ticket_price_checker":
            arguments = json.loads(tool_call.function.arguments)
            destination_city = arguments.get("destination_city", "")
            print(f"Tool call arguments: {arguments}")
            price = ticket_price_checker(destination_city)
            
            response = {
                "role": "tool",
                "content": price,
                "tool_call_id": tool_call.id
                }
            responses.append(response)
        else:
            responses.append({
                "role": "tool",
                "content": "Unknown tool called.",
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
def tooling_chat_v3(message, history):

    print("Inside the tooling_chat function")
    messages = [{"role": "system", "content": system_prompt}]
    
    for h in history:
        content = h["content"]
        if isinstance(content, list):
            content = content[0]["text"] if content else ""
        messages.append({
            "role": h["role"],
            "content": content
        })
    
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=curr_model,
        messages=messages,
        temperature=0,
        tools=tools,
        tool_choice="auto"
    )
    
    while response.choices[0].finish_reason == "tool_calls":
        messages.append({
        "role": "assistant",
        "content": response.choices[0].message.content,
        "tool_calls": response.choices[0].message.tool_calls
        })
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.extend(responses)

        for msg in messages:
            print(msg)

        response = client.chat.completions.create(
            model=curr_model,
            messages=messages,
            tools=tools,
        )

    return response.choices[0].message.content

In [119]:
gr.ChatInterface(fn=tooling_chat_v3).launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


Inside the tooling_chat function
Inside the tooling_chat function
Inside the tooling_chat function
Handling tool calls...
Message received for tool call: 
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_m8Ew4R3OBfYgOR1XZfSeBtip', function=Function(arguments='{"destination_city":"London"}', name='ticket_price_checker'), type='function')])
Tool call arguments: {'destination_city': 'London'}
Inside the ticket price checker function
Received destination city: London
{'role': 'system', 'content': '\nYou are a customer support assistant for an airline called Flight AI.\nYour only capability is to check ticket prices for destinations.\n\nRules:\n1. Respond in no more than one short, courteous sentence.\n2. If the user greeting is generic (e.g., "Hi", "Hello", "Hey"), respond with: "Hi, I am your Flight Assistant, how may I help you?"\n3. If the user asks about ticket 